# Unsupervised Text Classification

Using LLMs to categorize responses seems to be pretty unreliable and I have been unsuccessful in getting the models to cleanly work. 
In this document, I discuss using `BERTopic` to create categories of responses. This seems to be a fairly popular method in recent years.



## Questions

All responses can be found in `data/student-responses.csv`. In this file, all collected responses are stored from the bar chart and heatmap experiments. In the data cleaning script, a flag for bar chart or heatmap experiment was included.

In [123]:
# Import modules
import pandas as pd

# Read dataset
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]
df.head()

,id,section_sis_id,section,attempt,What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract,Paste the response code you received after participating in the graphics experiment here,As of today I am at least 19 years of age,My instructor may share my reflection responses with the researchers in this study,What do you think the purpose of the experiment was,What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why,...,What sources of error are involved in this experiment,What variables were examined For each variable identify whether it was quantitative or categorical,In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens,How did the information you gained from the components of this project participation post study reflection extended abstract presentation differ,If you had to hear about this study using only the extended abstract or only the presentation which one would you prefer Which one would be better for determining whether the experiment was well designed,What components were emphasized in the presentation that weren t emphasized in the abstract Why do you think that is,What critiques do you have of this study and its design What would have made the study better,year,semester,experiment
0,15464f12481fd2057f7a8205a42e03e8,STAT-218-011.1241,1,1,Seeing the marked bar closer and in person rat...,40646-92578-95520-71653,True,I agree,To see how accurate the students can be with e...,I think this was a paried control group becaus...,...,There's no results in whether if you were clos...,Circle and Triangle graph. Quantitative,I think that’ll look a lot different to a pers...,They gave us charts and information that shows...,"Extended abstract, extended abstarct",2d and 3d bar graphs because they were trying ...,How do you know whether or not your response i...,2024,Spring,Bar chart
1,2800ab155946ad01714a453f448edbae,STAT-218-011.1241,1,1,I think it is clearer that this experiment is ...,53670-79351-03305-70518,True,I agree,I think the purpose was to determine is we cou...,I think the control group was the models we ha...,...,"It might be a sampling error, also people may ...",Model size- quantitative color- categorical,A researcher has to form a hypothesis and crea...,I think the presentation was the easiest to un...,I would prefer to hear about the study from th...,I think the presentation emphasized more of th...,I think this study has a big possibility for c...,2024,Spring,Bar chart
2,43c3004415a1e8390ac4e7df8d06a292,STAT-218-011.1241,1,1,The whole objective of the experiment is more ...,13531-97095-57767-17258,True,I agree,The purpose of the experiment was to determine...,Randomization was used because we were given d...,...,The people who did the experiment could have m...,The height which was quantitative and the othe...,The process of scientific investigation is jus...,During the participation and post-study reflec...,I would prefer the video just because abstract...,The component that was emphasized to me was th...,Maybe to explain the experiment better to the ...,2024,Spring,Bar chart
3,629669a5f6966aac68902888966f15be,STAT-218-011.1241,1,1,NaN,12068-97156-84567-38829,True,I agree,NaN,NaN,...,NaN,NaN,I believe that it will consist of the person b...,NaN,NaN,NaN,NaN,2024,Spring,Bar chart
4,7b3570d08b16b612f899afa0a3bc4801,NaN,1,1,NaN,NaN,True,I agree,NaN,NaN,...,NaN,NaN,I believe that scientific investigation is the...,NaN,NaN,NaN,NaN,2024,Spring,Bar chart


In [124]:
df["section"].value_counts().to_frame(name="count").reset_index().rename(columns={"index": "section"}).sort_values("section")

,section,count
29,1,9
27,2,10
1,3,47
25,4,13
28,5,10
20,6,16
21,7,15
11,8,20
4,9,38
8,10,23


Next, it is worth noting that the quesitons are not in order. Below is a table summary of the questions and their corresponding modules.

| Module | Question Number | Dataframe Column Index (0-index) | Prompt |
|---|---:|---:|---|
| pre-experiment | Q1 | 13 | In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens. |
| post-experiment | Q2 | 8 | What do you think the purpose of the experiment was? |
| post-experiment | Q3 | 10 | What hypotheses might the experimenter have been testing? |
| post-experiment | Q4 | 11 | What sources of error are involved in this experiment? |
| post-experiment | Q5 | 12 | What variables were examined? For each variable, identify whether it was quantitative or categorical. |
| post-experiment | Q6 | 9 | What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why? |
| abstract reflection | Q7 | 4 | What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract. |
| presentation reflection | Q8 | 14 | How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ? |
| presentation reflection | Q9 | 16 | What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is? |
| presentation reflection | Q10 | 17 | What critiques do you have of this study and its design? What would have made the study better? |
| presentation reflection | Q11 | 15 | If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed? |

## Side note

It looks like some students may have used AI in their answers. While I doubt that we could 

- ac6eb19c384b5ec70af8f6b086758376: this user had some answers talking about cows, which is completely irrelevant to the sources of error question

## BERTopic

BERTopic is an unsupervised text classification method that leverages clustering and dimensional reduction. 

**Webpage:** <https://maartengr.github.io/BERTopic/index.html>

1. 

**Guide:** <https://www.youtube.com/watch?v=v3SePt3fr9g>

In [125]:
# Modules for topic modeling
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
import openai
from bertopic.representation import OpenAI
from bertopic.representation import KeyBERTInspired

# UMAP parameters and seed for reproducibility (I think these are the defaults, but I mostly wanted to set the random state)
umap_model = UMAP(n_neighbors=15, 
                  n_components=5, 
                  min_dist=0.0, 
                  metric='cosine', 
                  random_state=42)


# Configure HDBSCAN to have more clusters
hdb = hdbscan.HDBSCAN(
    min_cluster_size=10,     # ↓ smaller = more clusters
    min_samples=2,          # ↓ more sensitive
    prediction_data=True,   # required for some BERTopic visualizations
    cluster_selection_epsilon=0.1  # encourages splitting
)

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

#representation_model = OpenAI(client, model='mistral')
representation_model = KeyBERTInspired()

# Fit model for Q1 responses
q1 = df.iloc[:, 15].dropna().astype(str).tolist()

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2", 
    umap_model=umap_model,
    #n_gram_range = (1, 3), # I doubt there will be n-grams with 5 words, but it may be helpful
    calculate_probabilities=True,
    zeroshot_topic_list= ["Prefer presentation", "Prefer abstract"],
    zeroshot_min_similarity=0.75,
    hdbscan_model=hdb,
    nr_topics="auto",
    verbose=True,
    representation_model=representation_model
)
topics, probs = topic_model.fit_transform(q1)


2026-04-06 22:24:36,485 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11213.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  6.93it/s]
2026-04-06 22:24:40,094 - BERTopic - Embedding - Completed ✓
2026-04-06 22:24:40,095 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:24:40,579 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:24:40,580 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-04-06 22:24:40,589 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-04-06 22:24:40,910 - BERTopic - Clu

In [126]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,42,-1_presentation_experiment_design_results,"[presentation, experiment, design, results, st...",[I prefered the presentation as it provided mo...
1,0,114,0_presentation_abstract_experiment_extended,"[presentation, abstract, experiment, extended,...",[I would prefer to hear about the study using ...
2,1,56,1_presentation_visuals_comprehend_study,"[presentation, visuals, comprehend, study, hea...","[Personally, I would watch the presentation. T..."
3,2,55,2_presentation_abstract_experiment_design,"[presentation, abstract, experiment, design, r...",[I prefer the presentation because it was easi...
4,3,44,3_presentation_abstract_design_visuals,"[presentation, abstract, design, visuals, disc...",[I think that the presentation would be better...
5,4,43,4_presentation_experiment_experimental_design,"[presentation, experiment, experimental, desig...",[I would prefer the presentation because it of...
6,5,30,5_experiment_presentation_design_designed,"[experiment, presentation, design, designed, r...",[I would prefer to hear about the study from t...
7,6,19,6_abstract_experiment_scientific_research,"[abstract, experiment, scientific, research, s...",[I would prefer the abstract. It better explai...
8,7,2,7_presentation_information_the_from,"[presentation, information, the, from, this, p...","[The presentation. The presentation. , I would..."


In [127]:
# Perform hierarchical topic reduction
hierarchical_topics = topic_model.hierarchical_topics(q1)

# Visualize the subtopics
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 7/7 [00:00<00:00, 12.60it/s]


In [128]:
topic_model.visualize_topics()

In [129]:
topic_model.visualize_documents(q1)

## Function that fits model for specified question

In [130]:
def fit_bertopic(df, column_index):
    """
    Fit BERTopic on one dataframe column.

    Returns:
        topic_model: fitted BERTopic model
        topics: topic assignment list for each document
        probs: topic probability matrix from BERTopic
        docs: list of documents used for fitting
    """
    # Pull selected column, keeping only non-empty responses
    prompt_label = str(df.columns[column_index])
    responses = df.iloc[:, column_index]
    valid_mask = responses.notna() & responses.astype(str).str.strip().ne("")
    docs = responses[valid_mask].astype(str).tolist()

    if len(docs) == 0:
        raise ValueError(f"No non-empty responses found in column index {column_index}.")

    print(f"Running BERTopic for prompt: {prompt_label}")

    # UMAP setup
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    # HDBSCAN setup
    hdb = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=2,
        prediction_data=True,
        cluster_selection_epsilon=0.1
    )

    # OpenAI/Ollama-backed representation model setup
    client = openai.OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    representation_model = OpenAI(client, model='mistral') #use LLM to generate topic representations
    #representation_model = KeyBERTInspired() #remove stop words from representation model

    # Fit BERTopic
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        umap_model=umap_model,
        n_gram_range=(1, 3),
        calculate_probabilities=True,
        hdbscan_model=hdb,
        verbose=True,
        nr_topics="auto",
        representation_model=representation_model
    )
    topics, probs = topic_model.fit_transform(docs)

    return topic_model, topics, probs, docs

In [131]:
# Example for Q1 responses (column index 13)
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


2026-04-06 22:24:44,495 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12964.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:05<00:00,  3.99it/s]
2026-04-06 22:24:51,305 - BERTopic - Embedding - Completed ✓
2026-04-06 22:24:51,306 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:24:52,072 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:24:52,073 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:24:52,112 - BERTopic - Cluster - Completed ✓
2026-04-06 22:24:52,113 - BERTopic - Representation - Extracting topics using c-TF-IDF for topi

In [132]:
q1_topic_model.visualize_topics()

In [133]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(df.iloc[:, 13].dropna().astype(str).tolist())
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

100%|██████████| 29/29 [00:54<00:00,  1.89s/it]


# Paper Draft

## Methods

**NOTE:** This section is only for these models. I have more in the actual manuscript.

Our sample is composed of students enrolled in STAT 218 at the University of Nebraska--Lincoln. While all students were required to participant in the experiential learning project as part of the course cirriculum, data was collected if students meet the age of majority in Nebraska (age 19 or older) and if they consented to data collection. The data collection took place between Summer 2023 and Spring 2025, where XX sections of STAT 218 participated. 


### Text Classification

In recent years, there has been a growing research area in Large-Language Models (LLMs) for classifying open-ended survey responses. Despite the promising aspect of using LLMs for classification, these models are sensitive to prompt and token limits (XXX), which can greatly influence the outputs. Additionally, these models tend to suffer from hallucinations and/or low accuracy rates compared to traditional human codings (XXX). The current capacity of LLMs are not yet ready for standalone classification, but they do have some integrations with non-zero box methods. 

For the classiciation of our responses, we focus on BERTopic (XXX), which is a topic modeling algorithm that incorporates clustering and dimensional reduction. BERTopic is highly customizable in each stage of the algorithm, allowing for multiple specifications for fine-tuning. Descriptions of this process can be found in Table (XXX), along with our specified models and hyper-parameters for each stage. We note a few of our chosen hyperparameter selections. For the clustering stage with HBDSCAN, we set the minimum cluster size to 5 so that smaller clusters can form. We found that the default settings were too restrictive and formed topics that closely aligned with the original prompt. We also included n-grams up to five words to allow for common phrases that students may have responded with (e.g., "scientific process"). Lastly, we incorporated Mistral (XXX) as a locally-run LLM to fine-tune the generated topic lists into interpretable categories, while also respecting concerns over data privacy of cloud-based LLMs.




| Stage | Step | Purpose | Option Used |
|---|---|---|---|
| Stage 1 | extract embeddings | Convert each response into a numeric vector that captures semantic meaning. | `embedding_model="all-MiniLM-L6-v2"` |
| Stage 2 | reduce dimensionality | Compress embeddings into a lower-dimensional space for better clustering efficiency | `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)` |
| Stage 3 | cluster reduced embeddings | Group similar responses into topic clusters. | `HDBSCAN(min_cluster_size=5, min_samples=2, cluster_selection_epsilon=0.1, prediction_data=True)` |
| Stage 4 | tokenize topics | Break text into candidate terms/phrases used to represent each cluster. | `n_gram_range=(1, 5)` |
| Stage 5 | extract topic words | Compute the most representative words/phrases for each cluster. | BERTopic default c-TF-IDF weighting |
| Stage 6 | fine-tune topic representations | Improve readability and specificity of topic labels/keywords. | `representation_model=OpenAI(client, model="mistral")` with Ollama endpoint `http://localhost:11434/v1` |

## Results

### Pre-Experiment

The pre-experiment prompt consisted of asking participants to write a paragraph about how science differs from researchers and the general public. XXX students provided a response to this prompt.

> In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens.


In [134]:
# Q1
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

2026-04-06 22:26:49,351 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11498.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:05<00:00,  3.99it/s]
2026-04-06 22:26:56,577 - BERTopic - Embedding - Completed ✓
2026-04-06 22:26:56,578 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:26:57,340 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:26:57,340 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:26:57,379 - BERTopic - Cluster - Completed ✓
2026-04-06 22:26:57,379 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-06 22:26:57,582 - BERTopic - Representation - Completed ✓
2026-04-

In [135]:
q1_topic_model.visualize_topics()

In [136]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(q1_docs)
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

100%|██████████| 29/29 [00:55<00:00,  1.91s/it]


### Post-Experiment

#### Q2

**Prompt:**
> What do you think the purpose of the experiment was?

In [137]:
# Q2
q2_topic_model, q2_topics, q2_probs, q2_docs = fit_bertopic(
    df=df,
    column_index=8
)

2026-04-06 22:28:48,064 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What do you think the purpose of the experiment was


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13365.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 13.41it/s]
2026-04-06 22:28:50,948 - BERTopic - Embedding - Completed ✓
2026-04-06 22:28:50,948 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:28:51,539 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:28:51,539 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:28:51,568 - BERTopic - Cluster - Completed ✓
2026-04-06 22:28:51,569 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-06 22:28:51,612 - BERTopic - Representation - Completed ✓
2026-04-

In [138]:
q2_topic_model.visualize_topics()

In [139]:
q2_hierarchical_topics = q2_topic_model.hierarchical_topics(q2_docs)
q2_topic_model.visualize_hierarchy(hierarchical_topics=q2_hierarchical_topics)

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:23<00:00,  1.17s/it]


#### Q3

**Prompt:**
> What hypotheses might the experimenter have been testing?

In [140]:
# Q3
q3_topic_model, q3_topics, q3_probs, q3_docs = fit_bertopic(
    df=df,
    column_index=10
)

Running BERTopic for prompt: What hypotheses might the experimenter have been testing


2026-04-06 22:29:43,532 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13204.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00,  9.62it/s]
2026-04-06 22:29:47,341 - BERTopic - Embedding - Completed ✓
2026-04-06 22:29:47,342 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:29:47,919 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:29:47,919 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:29:47,960 - BERTopic - Cluster - Completed ✓
2026-04-06 22:29:47,961 - BERTopic - Representation - Extracting topics using c-TF-IDF for topi

In [141]:
q3_topic_model.visualize_topics()

In [142]:
q3_hierarchical_topics = q3_topic_model.hierarchical_topics(q3_docs)
q3_topic_model.visualize_hierarchy(hierarchical_topics=q3_hierarchical_topics)

100%|██████████| 36/36 [00:48<00:00,  1.36s/it]


#### Q4

**Prompt:**
> What sources of error are involved in this experiment?

In [143]:
# Q4
q4_topic_model, q4_topics, q4_probs, q4_docs = fit_bertopic(
    df=df,
    column_index=11
)

Running BERTopic for prompt: What sources of error are involved in this experiment


2026-04-06 22:31:25,269 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11728.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 10.77it/s]
2026-04-06 22:31:28,550 - BERTopic - Embedding - Completed ✓
2026-04-06 22:31:28,550 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:31:29,132 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:31:29,133 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:31:29,165 - BERTopic - Cluster - Completed ✓
2026-04-06 22:31:29,166 - BERTopic - Representation - Extracting topics using c-TF-IDF for topi

In [144]:
q4_topic_model.visualize_topics()

In [145]:
q4_hierarchical_topics = q4_topic_model.hierarchical_topics(q4_docs)
q4_topic_model.visualize_hierarchy(hierarchical_topics=q4_hierarchical_topics)

100%|██████████| 31/31 [00:39<00:00,  1.26s/it]


#### Q5

**Prompt:**
> What variables were examined? For each variable, identify whether it was quantitative or categorical.

In [146]:
# Q5
q5_topic_model, q5_topics, q5_probs, q5_docs = fit_bertopic(
    df=df,
    column_index=12
)

Running BERTopic for prompt: What variables were examined For each variable identify whether it was quantitative or categorical


2026-04-06 22:32:48,270 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14048.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00, 10.53it/s]
2026-04-06 22:32:52,015 - BERTopic - Embedding - Completed ✓
2026-04-06 22:32:52,015 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 22:32:52,583 - BERTopic - Dimensionality - Completed ✓
2026-04-06 22:32:52,584 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 22:32:52,618 - BERTopic - Cluster - Completed ✓
2026-04-06 22:32:52,619 - BERTopic - Representation - Extracting topics using c-TF-IDF for topi

In [147]:
q5_topic_model.visualize_topics()

In [148]:
q5_hierarchical_topics = q5_topic_model.hierarchical_topics(q5_docs)
q5_topic_model.visualize_hierarchy(hierarchical_topics=q5_hierarchical_topics)

100%|██████████| 23/23 [00:27<00:00,  1.21s/it]


In [149]:
q5_topic_model.visualize_document_datamap(q5_docs)

NameError: name 'datamapplot' is not defined

#### Q6

**Prompt:**
> What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why?

In [ ]:
# Q6
q6_topic_model, q6_topics, q6_probs, q6_docs = fit_bertopic(
    df=df,
    column_index=9
)

Running BERTopic for prompt: What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why


2026-04-06 17:44:21,100 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13747.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:01<00:00,  8.78it/s]
2026-04-06 17:44:24,671 - BERTopic - Embedding - Completed ✓
2026-04-06 17:44:24,671 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:44:25,226 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:44:25,227 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:44:25,254 - BERTopic - Cluster - Completed ✓
2026-04-06 17:44:25,255 - BERTopic - Representation - Fine-tuning topics using representation m

In [ ]:
q6_topic_model.visualize_topics()

In [ ]:
q6_hierarchical_topics = q6_topic_model.hierarchical_topics(q6_docs)
q6_topic_model.visualize_hierarchy(hierarchical_topics=q6_hierarchical_topics)

100%|██████████| 33/33 [00:37<00:00,  1.14s/it]


### Abstract Reflection

#### Q7

**Prompt:**
> What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract.

In [ ]:
# Q7
q7_topic_model, q7_topics, q7_probs, q7_docs = fit_bertopic(
    df=df,
    column_index=4
)

2026-04-06 17:45:41,021 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13010.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 16/16 [00:03<00:00,  4.52it/s]
2026-04-06 17:45:46,234 - BERTopic - Embedding - Completed ✓
2026-04-06 17:45:46,235 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:45:46,760 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:45:46,761 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:45:46,791 - BERTopic - Cluster - Completed ✓
2026-04-06 17:45:46,792 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 40/40 [01:01<00:00,  1.55s/it]
2026-04-06 17:46:49,184 - BERTop

In [ ]:
q7_topic_model.visualize_topics()

In [ ]:
q7_hierarchical_topics = q7_topic_model.hierarchical_topics(q7_docs)
q7_topic_model.visualize_hierarchy(hierarchical_topics=q7_hierarchical_topics)

100%|██████████| 38/38 [01:05<00:00,  1.73s/it]


### Presentation Reflection

#### Q8

**Prompt:**
> How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ?

In [ ]:
# Q8
q8_topic_model, q8_topics, q8_probs, q8_docs = fit_bertopic(
    df=df,
    column_index=14
)

2026-04-06 17:47:55,445 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: How did the information you gained from the components of this project participation post study reflection extended abstract presentation differ


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13339.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:02<00:00,  6.29it/s]
2026-04-06 17:47:59,219 - BERTopic - Embedding - Completed ✓
2026-04-06 17:47:59,219 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:47:59,629 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:47:59,630 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:47:59,648 - BERTopic - Cluster - Completed ✓
2026-04-06 17:47:59,650 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 26/26 [00:28<00:00,  1.08s/it]
2026-04-06 17:48:28,050 - BERTop

In [ ]:
q8_topic_model.visualize_topics()

In [ ]:
q8_hierarchical_topics = q8_topic_model.hierarchical_topics(q8_docs)
q8_topic_model.visualize_hierarchy(hierarchical_topics=q8_hierarchical_topics)

100%|██████████| 24/24 [00:27<00:00,  1.13s/it]


#### Q9

**Prompt:**
> What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is?

In [ ]:
# Q9
q9_topic_model, q9_topics, q9_probs, q9_docs = fit_bertopic(
    df=df,
    column_index=16
)

2026-04-06 17:48:55,503 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: What components were emphasized in the presentation that weren t emphasized in the abstract Why do you think that is


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14031.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.73it/s]
2026-04-06 17:48:58,763 - BERTopic - Embedding - Completed ✓
2026-04-06 17:48:58,764 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:48:59,168 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:48:59,169 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:48:59,189 - BERTopic - Cluster - Completed ✓
2026-04-06 17:48:59,191 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 30/30 [00:31<00:00,  1.06s/it]
2026-04-06 17:49:31,314 - BERTop

In [ ]:
q9_topic_model.visualize_topics()

In [ ]:
q9_hierarchical_topics = q9_topic_model.hierarchical_topics(q9_docs)
q9_topic_model.visualize_hierarchy(hierarchical_topics=q9_hierarchical_topics)

100%|██████████| 28/28 [00:29<00:00,  1.06s/it]


#### Q10

**Prompt:**
> What critiques do you have of this study and its design? What would have made the study better?

In [ ]:
# Q10
q10_topic_model, q10_topics, q10_probs, q10_docs = fit_bertopic(
    df=df,
    column_index=17
)

Running BERTopic for prompt: What critiques do you have of this study and its design What would have made the study better


2026-04-06 17:50:01,385 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13314.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.58it/s]
2026-04-06 17:50:04,680 - BERTopic - Embedding - Completed ✓
2026-04-06 17:50:04,681 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:50:05,096 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:50:05,096 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:50:05,114 - BERTopic - Cluster - Completed ✓
2026-04-06 17:50:05,116 - BERTopic - Representation - Fine-tuning topics using representation m

In [ ]:
q10_topic_model.visualize_topics()

In [ ]:
q10_hierarchical_topics = q10_topic_model.hierarchical_topics(q10_docs)
q10_topic_model.visualize_hierarchy(hierarchical_topics=q10_hierarchical_topics)

100%|██████████| 23/23 [00:21<00:00,  1.08it/s]


#### Q11

**Prompt:**
> If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed?

In [ ]:
# Q11
q11_topic_model, q11_topics, q11_probs, q11_docs = fit_bertopic(
    df=df,
    column_index=15
)

Running BERTopic for prompt: If you had to hear about this study using only the extended abstract or only the presentation which one would you prefer Which one would be better for determining whether the experiment was well designed


2026-04-06 17:50:52,116 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11476.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:01<00:00,  7.09it/s]
2026-04-06 17:50:55,607 - BERTopic - Embedding - Completed ✓
2026-04-06 17:50:55,607 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-06 17:50:56,016 - BERTopic - Dimensionality - Completed ✓
2026-04-06 17:50:56,017 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-06 17:50:56,034 - BERTopic - Cluster - Completed ✓
2026-04-06 17:50:56,036 - BERTopic - Representation - Fine-tuning topics using representation m

In [ ]:
q11_topic_model.visualize_topics()

In [ ]:
q11_hierarchical_topics = q11_topic_model.hierarchical_topics(q11_docs)
q11_topic_model.visualize_hierarchy(hierarchical_topics=q11_hierarchical_topics)

100%|██████████| 23/23 [00:20<00:00,  1.14it/s]
